# MentalBERT – Proof of Concept

This notebook demonstrates a **proof-of-concept** pipeline for fine-tuning
[MentalBERT](https://huggingface.co/mental/mental-bert-base-uncased) on a
mental-health symptom classification task using **pseudo (synthetic) data**.

### Pipeline overview
1. Generate synthetic labelled examples (Depression, Anxiety, Bipolar, Normal)
2. Load MentalBERT tokenizer & model (with `bert-base-uncased` fallback if gated access is unavailable)
3. Tokenize & build `Dataset` objects
4. Fine-tune a `BertForSequenceClassification` head
5. Evaluate on a held-out pseudo test set

> **Note:** MentalBERT is a *gated* model on HuggingFace.  
> If you have access, run `huggingface-cli login` first.  
> Otherwise the notebook falls back to `bert-base-uncased`.

## 0 · Imports & Configuration

In [3]:
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
import evaluate
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


## 1 · Generate Pseudo Data

We create synthetic examples that *mimic* social-media posts across four
mental-health categories.  Each category has a pool of template sentences;
we randomly sample and lightly perturb them to create variety.

In [5]:
LABEL2ID = {"Normal": 0, "Depression": 1, "Anxiety": 2, "Bipolar": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

# ── template pools ──────────────────────────────────────────────────
TEMPLATES = {
    "Normal": [
        "Had a great day at work today, feeling productive and happy.",
        "Just finished a nice workout, feeling energized.",
        "Spent quality time with friends over coffee this afternoon.",
        "Looking forward to the weekend trip we planned.",
        "The weather is beautiful today, went for a walk in the park.",
        "Cooked a new recipe tonight and it turned out amazing.",
        "Had a really interesting conversation with my coworker today.",
        "Feeling grateful for the small things in life.",
        "Watched a funny movie with my family, we laughed so much.",
        "Finally finished that book I have been reading, loved the ending.",
    ],
    "Depression": [
        "I feel empty inside, nothing seems to matter anymore.",
        "I can not get out of bed, everything feels so heavy.",
        "I have lost interest in things I used to enjoy.",
        "I feel like a burden to everyone around me.",
        "Nothing makes me happy, I just feel numb all the time.",
        "I have been sleeping all day but still feel exhausted.",
        "I do not see the point in trying anymore, everything feels hopeless.",
        "I cry for no reason and I cannot stop.",
        "I feel worthless and like I will never be good enough.",
        "Every day feels the same, just going through the motions.",
    ],
    "Anxiety": [
        "My heart is racing and I cannot calm down.",
        "I keep worrying about things that have not happened yet.",
        "I feel like something terrible is about to happen.",
        "I cannot stop overthinking every little decision.",
        "I had a panic attack at the grocery store today.",
        "I am constantly on edge, my muscles are so tense.",
        "I avoid social situations because they make me so nervous.",
        "I woke up at 3am with racing thoughts and could not fall back asleep.",
        "What if I fail? What if everything goes wrong? I cannot stop these thoughts.",
        "I feel dizzy and short of breath even though nothing is wrong.",
    ],
    "Bipolar": [
        "I went from feeling on top of the world to completely crashing in a few hours.",
        "I spent way too much money last week during a manic episode.",
        "Some days I have so much energy I do not need sleep, other days I cannot move.",
        "My mood swings are ruining my relationships.",
        "I feel invincible one moment and completely worthless the next.",
        "I made impulsive decisions I deeply regret now that the high is over.",
        "My thoughts race so fast I cannot keep up, then everything goes silent.",
        "People say I am unpredictable but I cannot control it.",
        "During my highs I start so many projects, during my lows I abandon them all.",
        "The cycling between extreme moods is exhausting.",
    ],
}

# ── augmentation helpers ────────────────────────────────────────────
FILLER_PHRASES = [
    "honestly ", "tbh ", "idk ", "like ", "seriously ",
    "you know ", "I mean ", "basically ", "really ", "",
]

def augment(text: str) -> str:
    """Light augmentation: prepend a random filler and optionally lower-case."""
    filler = random.choice(FILLER_PHRASES)
    text = filler + text
    if random.random() < 0.3:
        text = text.lower()
    return text

# ── generate dataset ────────────────────────────────────────────────
N_PER_CLASS = 80  # 80 × 4 = 320 total samples

rows = []
for label_name, templates in TEMPLATES.items():
    for _ in range(N_PER_CLASS):
        text = augment(random.choice(templates))
        rows.append({"text": text, "label": LABEL2ID[label_name]})

df = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Total samples: {len(df)}")
print(f"\nLabel distribution:\n{df['label'].map(ID2LABEL).value_counts()}")
df.head(10)

Total samples: 320

Label distribution:
label
Anxiety       80
Normal        80
Depression    80
Bipolar       80
Name: count, dtype: int64


,text,label
0,you know I feel dizzy and short of breath even...,2
1,basically I avoid social situations because th...,2
2,"like Had a great day at work today, feeling pr...",0
3,"like had a great day at work today, feeling pr...",0
4,seriously cooked a new recipe tonight and it t...,0
5,"really Nothing makes me happy, I just feel num...",1
6,What if I fail? What if everything goes wrong?...,2
7,"you know I can not get out of bed, everything ...",1
8,like i spent way too much money last week duri...,3
9,I feel dizzy and short of breath even though n...,2


## 2 · Train / Validation / Test Split

In [6]:
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED
)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")

Train: 224  |  Val: 48  |  Test: 48


## 3 · Load MentalBERT (with fallback)

We first try to load `mental/mental-bert-base-uncased`.  
If access is denied (gated model), we fall back to `bert-base-uncased`.

In [7]:
MODEL_NAME = "mental/mental-bert-base-uncased"
FALLBACK_MODEL = "bert-base-uncased"

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )
    print(f"✅ Loaded gated model: {MODEL_NAME}")
except Exception as e:
    print(f"⚠️  Could not load {MODEL_NAME}: {e}")
    print(f"↪ Falling back to {FALLBACK_MODEL}")
    MODEL_NAME = FALLBACK_MODEL
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )
    print(f"✅ Loaded fallback model: {MODEL_NAME}")

model.to(DEVICE)
print(f"\nModel parameters: {model.num_parameters():,}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at mental/mental-bert-base-uncased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Loaded gated model: mental/mental-bert-base-uncased

Model parameters: 109,485,316


## 4 · Tokenize & Build HF Datasets

In [8]:
MAX_LEN = 128

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
    )

train_ds = Dataset.from_pandas(train_df).map(tokenize_fn, batched=True)
val_ds   = Dataset.from_pandas(val_df).map(tokenize_fn, batched=True)
test_ds  = Dataset.from_pandas(test_df).map(tokenize_fn, batched=True)

# Set format for PyTorch
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format(type="torch",   columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch",  columns=["input_ids", "attention_mask", "label"])

print(f"Train features: {train_ds.features}")
print(f"Sample input_ids shape: {train_ds[0]['input_ids'].shape}")

Map: 100%|██████████| 48/48 [00:00<00:00, 3472.47 examples/s]

Train features: {'text': Value('string'), 'label': Value('int64'), '__index_level_0__': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}
Sample input_ids shape: torch.Size([128])


## 5 · Fine-Tune with 🤗 Trainer

In [9]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)
    f1  = f1_metric.compute(predictions=preds, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

training_args = TrainingArguments(
    output_dir="./results/mentalbert_poc",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=10,
    seed=SEED,
    report_to="none",  # disable W&B / MLflow for POC
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.260700,0.889384,0.979167,0.979130
2,0.942600,0.469074,1.000000,1.000000
3,0.386200,0.242402,1.000000,1.000000
4,0.262600,0.142218,1.000000,1.000000
5,0.159300,0.116292,1.000000,1.000000


TrainOutput(global_step=70, training_loss=0.5448085393224443, metrics={'train_runtime': 390.7226, 'train_samples_per_second': 2.866, 'train_steps_per_second': 0.179, 'total_flos': 73672418426880.0, 'train_loss': 0.5448085393224443, 'epoch': 5.0})

## 6 · Evaluate on Test Set

In [10]:
test_results = trainer.evaluate(test_ds)
print("\n── Test-set metrics ──")
for k, v in test_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


── Test-set metrics ──
  eval_loss: 0.5278
  eval_accuracy: 0.9792
  eval_f1: 0.9791
  eval_runtime: 4.5784
  eval_samples_per_second: 10.4840
  eval_steps_per_second: 0.4370
  epoch: 5.0000


## 7 · Detailed Classification Report & Confusion Matrix

In [ ]:
# Get predictions on the test set
preds_output = trainer.predict(test_ds)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = test_df["label"].values

# Classification report
target_names = [ID2LABEL[i] for i in range(NUM_LABELS)]
print(classification_report(y_true, y_pred, target_names=target_names))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
print("Confusion Matrix:")
cm_df

## 8 · Interactive Inference Demo

Pass arbitrary text to the fine-tuned model to see predictions.

In [ ]:
def predict(text: str) -> dict:
    """Return predicted label and class probabilities for a single text."""
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
    ).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    pred_id = int(np.argmax(probs))
    return {
        "predicted_label": ID2LABEL[pred_id],
        "confidence": float(probs[pred_id]),
        "probabilities": {ID2LABEL[i]: round(float(p), 4) for i, p in enumerate(probs)},
    }

# ── demo predictions ────────────────────────────────────────────────
demo_texts = [
    "I feel so hopeless and I cannot stop crying.",
    "My heart is pounding and I cannot breathe properly.",
    "Had a wonderful dinner with family tonight!",
    "One minute I am full of energy, the next I crash completely.",
]

for text in demo_texts:
    result = predict(text)
    print(f"\n📝 \"{text}\"")
    print(f"   → {result['predicted_label']} (confidence: {result['confidence']:.2%})")
    print(f"   → probabilities: {result['probabilities']}")

---
### Next Steps
- Obtain gated access to `mental/mental-bert-base-uncased` and re-run
- Replace pseudo data with the real symptom dataset
- Experiment with hyperparameters (learning rate, epochs, batch size)
- Add cross-validation for more robust evaluation
- Compare against baseline models (TF-IDF + LogReg, dummy classifiers)